# Create Erasmus+ Awards from the Erasmus+ Project Results platform

Creates Erasmus+ awards from the EU's Erasmus+ Project Results platform
(https://erasmus-plus.ec.europa.eu/projects) via its VALOR search service
(`POST https://ec.europa.eu/programmes/service/search/project/search`, open, no auth).
~327K funded projects covering both programme periods (2014-2020 and 2021-2027).

A bulk file was checked FIRST per the ingest-method ladder: data.europa.eu carries no
central Erasmus+ *projects* dataset (only mobility statistics and per-university lists),
so the platform's own search API is the authoritative source.

**Prerequisites:**
- Run `scripts/local/erasmus_plus_to_s3.py` to download and upload the data first.

**Data source:** https://erasmus-plus.ec.europa.eu/projects (VALOR search API)
**S3 location:** `s3a://openalex-ingest/awards/erasmus_plus/erasmus_plus_projects.parquet`

**Erasmus+ funder:**
- funder_id: 4320335551
- display_name: "Erasmus+"
- ROR: null (programme, not an organisation — no ROR)
- DOI: 10.13039/501100010790

**Amount/currency:** EUR, implicit (EU programme); source field `projectGrantedEuAmount`
("EU grant"), ~100% coverage.

**Lead investigator:** the platform publishes **organisations only** (coordinator +
partners), no person PI -> person fields are source-authority NULL; the coordinator
organisation ships in `lead_investigator.affiliation` (name + country). Partners are
kept in the raw table (`partners_json`) for future use.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet
CREATE OR REPLACE TABLE openalex.awards.erasmus_plus_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/erasmus_plus/erasmus_plus_projects.parquet`;

In [ ]:
%sql
-- Check row count (should be ~327K)
SELECT COUNT(*) as total_projects FROM openalex.awards.erasmus_plus_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.erasmus_plus_raw LIMIT 5;

In [ ]:
%sql
-- Check column names before writing transformation SQL (runbook 2.2 step 1.5)
DESCRIBE openalex.awards.erasmus_plus_raw;

## Step 1.6: Funder existence check (Path A — F4320* funder, must return exactly 1 row)

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320335551;  -- Erasmus+

## Step 2: Create Erasmus+ Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.erasmus_plus_awards
USING delta
AS
WITH
-- Get Erasmus+ funder from OpenAlex by explicit funder_id (Path A: F4320* is in the dim)
eplus_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320335551  -- Erasmus+
),

awards_transformed AS (
    SELECT
        -- Generate unique ID using xxhash64 of funder_id:project_reference
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.project_reference)))) % 9000000000 as id,

        -- Display name = project title
        g.project_title as display_name,

        -- Description: search summary, else the objectives / background sections
        COALESCE(g.project_description, g.description_objectives, g.description_background) as description,

        -- Funder info
        f.funder_id,
        g.project_reference as funder_award_id,

        -- Amount in EUR (projectGrantedEuAmount = EU grant)
        TRY_CAST(g.granted_eu_amount AS DOUBLE) as amount,
        'EUR' as currency,

        -- Funder struct
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type: Erasmus+ projects are programme grants
        'grant' as funding_type,

        -- Funder scheme = action type (level3), falling back to key action (level2)
        COALESCE(g.level3_label, g.level2_label) as funder_scheme,

        -- Provenance
        'erasmus_plus' as provenance,

        -- Dates (stored as strings in YYYY-MM-DD format)
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,

        -- Lead investigator: no person PI in the source (organisations only) --
        -- person fields are source-authority NULL; coordinator org = affiliation
        CASE
            WHEN g.coordinator_name IS NOT NULL THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.coordinator_name as name,
                        g.coordinator_country as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        -- Co-lead and other investigators (not available in the source)
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- Landing page URL = project card on the results platform
        g.landing_page_url as landing_page_url,

        -- No DOI for Erasmus+ projects
        CAST(NULL AS STRING) as doi,

        -- Works API URL
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.project_reference)))) % 9000000000) as works_api_url,

        -- Timestamps
        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.erasmus_plus_raw g
    CROSS JOIN eplus_funder f
    WHERE g.project_reference IS NOT NULL
      AND TRIM(g.project_reference) != ''
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'erasmus_plus' AND priority = 427;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    427 as priority  -- Erasmus+ priority
FROM openalex.awards.erasmus_plus_awards;

## Verification Queries

In [ ]:
%sql
-- 6.1 Basic count (should match the parquet row count, ~327K)
SELECT COUNT(*) as total_erasmus_plus_awards FROM openalex.awards.erasmus_plus_awards;

In [ ]:
%sql
-- 6.2 Schema validation
DESCRIBE openalex.awards.erasmus_plus_awards;

In [ ]:
%sql
-- 6.3 Data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_coordinator,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_dates
FROM openalex.awards.erasmus_plus_awards;

In [ ]:
%sql
-- 6.4 Sample inspection
SELECT id, display_name, funder_award_id, funder_scheme, funding_type,
       amount, currency, start_date, end_date,
       lead_investigator.affiliation.name as coordinator,
       lead_investigator.affiliation.country as coordinator_country
FROM openalex.awards.erasmus_plus_awards LIMIT 10;

In [ ]:
%sql
-- 6.4a display_name / coordinator frequency check.
-- Person-name check is n/a (person fields are NULL by design — see header).
-- Coordinators legitimately repeat (big National Agencies / universities run
-- many mobility projects); titles should show a real long-tail.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.erasmus_plus_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.5 Funder consistency (single funder expected: Erasmus+)
SELECT funder.display_name, funder_id, COUNT(*)
FROM openalex.awards.erasmus_plus_awards
GROUP BY funder.display_name, funder_id ORDER BY 3 DESC;

In [ ]:
%sql
-- 6.6 Year distribution (call years 2014-2026 expected)
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.erasmus_plus_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
-- 6.7 Amount and currency coverage (expected ~100%, EUR only)
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.erasmus_plus_awards;

In [ ]:
%sql
-- 6.8 Confirm the rows reached the shared raw table
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'erasmus_plus'
GROUP BY provenance, priority;